# 32 · Retrain, gates, promotion and rollback: changing the model on purpose

**Who this is for:** you, after notebook 30 (the drift check) and notebook 20 (training and aliases).
**Runs on:** Databricks serverless (costs credits: keep it short, detach when done). Most cells only read. Sections 3 and 4 need a retrain to have run in the catalog you pick: prod after its first approved retrain, staging after the retrain dry run.

**By the end you will be able to:**
- follow one incident through the tables: drift check, retrain request, retrain, promotion
- say what a retrained model must show to become `@challenger`, and why each gate is there
- explain why both models get the same alert budget in the comparison, and why whole stays are resampled
- say what a promotion and a rollback change (three aliases) and what they never change (hours already scored)
- read `ml.ops_audit`: who decided what, when and why

The decisions are made on GitHub (`.github/workflows/retrain.yml`, `promote.yml`, `rollback.yml`: an approval or a rejection, with a comment). The code that carries them out is `heavden.ml.training` (`compare_with_champion`) and `heavden.ml.model_ops`, tested locally.

## 1. Setup

Section 4 loads two model versions, so LightGBM is installed as in the jobs.

In [ ]:
%pip install lightgbm==4.6.0 --quiet

In [ ]:
import sys

# Find the repo folder this notebook was deployed into, and import the real code from it.
notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
repo_root = "/Workspace" + notebook_path.split("/notebooks/databricks/")[0]
sys.path.insert(0, f"{repo_root}/databricks/src")

import mlflow
import pandas as pd

from heavden.ml import evaluate, model_ops, tracking, training
from heavden.ml.training import to_utc

dbutils.widgets.text("catalog", "heavden_staging")
CATALOG = dbutils.widgets.get("catalog")
MODEL = f"{CATALOG}.ml.deterioration_risk"
spark.conf.set("spark.sql.session.timeZone", "UTC")
mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()
print(repo_root, MODEL)

## 2. Which model is which

Three aliases on the registered model are the whole deployment:

| Alias | Meaning | Who moves it |
|---|---|---|
| `@champion` | the model in use: batch scoring and the drift check load it | a promotion or a rollback, each approved on GitHub |
| `@challenger` | a retrained model that passed every gate and waits for a decision | the training job |
| `@previous_champion` | the champion before the last promotion or rollback | the same job that moves `@champion` |

Each training run also records its **data window** (the first and last hour of its training, validation and test blocks). A retrained model's blocks are counted back from the day it was trained, so they can't be rebuilt from job settings; scoring and the drift check read them from here.

In [ ]:
state = model_ops.status(client, MODEL)
print("aliases:", state["aliases"])

recorded = ("mode", "compared_with_version", "train_from", "valid_from", "test_from", "test_to")
versions = []
for v in state["versions"]:
    run = client.get_run(client.get_model_version(MODEL, v["version"]).run_id)
    versions.append(v | {k: run.data.tags.get(k) for k in recorded})
versions = pd.DataFrame(versions)
display(versions.tail(10))

## 3. The gates a retrain has to pass

A retrain (job `train` with `mode=retrain`, started by an approved retrain request) keeps the latest 2 days as test and the 2 days before as validation; all earlier history trains. The new model becomes `@challenger` only if it passes two sets of gates:

1. **Against NEWS2, on validation** (as for the first model): a higher AUPRC, and at least as many escalations flagged within the alert budget.
2. **Against the champion, on the test days**, with both models held to the same alert budget (the champion gets a fresh threshold from the same validation days, so the comparison is about the models, not about how old a threshold is):

| Gate | Needs | Why |
|---|---|---|
| Escalations flagged | at least as many, in at least 75% of resamples | what the ward gets: more deteriorations caught for the same number of alerts |
| AUROC | higher, in at least 75% of resamples | the ranking of all patients is better, not only at one threshold |
| AUPRC | higher in at least 10% of resamples | a guard: not clearly worse at the top of the ranking |
| Brier score | at most 10% worse | the risks shown to nurses stay honest |

**Why resamples, and of whole stays?** Two days hold 20 to 30 escalations, so one comparison ("0.21 beats 0.17") proves little. Both models are scored on the same 300 resamples of the test block. Whole stays are resampled, because a patient's hours move together: resampling single rows would make the result look more certain than it is.

**Why not simply "higher AUPRC"?** That was the first version of the gate. In an offline rehearsal after the protocol change, retrains flagged more escalations at the same alert budget and had the higher AUROC in nearly every run, while AUPRC moved either way: AUPRC is set by the very top of the ranking, where both models place the classic deteriorations, and what the retrain learns (the new protocol's reviews of mildly abnormal patients) sits lower down.

The numbers the gates looked at are stored on the retrain's MLflow run:

In [ ]:
retrains = versions[versions["mode"] == "retrain"]
if retrains.empty:
    print(f"No retrain has run in {CATALOG} yet: sections 3 and 4 have nothing to show.")
else:
    newest = retrains.iloc[-1]
    run_id = client.get_model_version(MODEL, newest["version"]).run_id
    metrics = client.get_run(run_id).data.metrics
    g = {
        k.removeprefix("vs_champion_"): v
        for k, v in metrics.items()
        if k.startswith("vs_champion_")
    }
    print(
        f"version {newest['version']} against version {newest['compared_with_version']}:",
        "passed" if g["beats_champion"] else "did not pass",
    )
    display(
        pd.DataFrame(
            [
                (
                    "escalations flagged",
                    g["escalations_flagged"],
                    g["champion_escalations_flagged"],
                    g["flagged_at_least_share"],
                    training.BETTER_SHARE,
                ),
                (
                    "AUROC",
                    g["auroc"],
                    g["champion_auroc"],
                    g["auroc_better_share"],
                    training.BETTER_SHARE,
                ),
                (
                    "AUPRC",
                    g["auprc"],
                    g["champion_auprc"],
                    g["auprc_better_share"],
                    training.NOT_WORSE_SHARE,
                ),
                ("Brier score (lower is better)", g["brier"], g["champion_brier"], None, None),
                (
                    "alerts per nurse per shift",
                    g["alerts_per_nurse_shift"],
                    g["champion_alerts_per_nurse_shift"],
                    None,
                    None,
                ),
            ],
            columns=[
                "metric",
                "new model",
                "champion",
                "share of resamples in its favour",
                "needs",
            ],
        ).round(3)
    )

## 4. The comparison, done again by hand

Nothing about the gates is hidden in the job: both versions are in the registry, the blocks are in the run's tags, and training sorts its rows, so the comparison can be repeated and gives the same numbers. (If hours were added to `training_exclusions.yaml` since that retrain, the blocks differ by those hours.)

In [ ]:
if not retrains.empty:
    new_version, old_version = newest["version"], newest["compared_with_version"]
    table = to_utc(
        spark.read.table(f"{CATALOG}.gold.training_set").toPandas(),
        ("prediction_ts", "label_known_at"),
    )
    table = training.drop_excluded(table, training.load_exclusions(CATALOG))
    table = table.sort_values(training.ROW_ORDER, ignore_index=True)  # the order training used
    blocks = tracking.data_blocks(MODEL, new_version)
    hours = table["prediction_ts"]
    valid, test = (table[hours.between(*blocks[b])] for b in ("valid", "test"))

    new_model = mlflow.sklearn.load_model(f"models:/{MODEL}/{new_version}")
    old_model = mlflow.sklearn.load_model(f"models:/{MODEL}/{old_version}")
    budget = evaluate.AlertBudget()
    thresholds = {
        "challenger": metrics["valid_threshold"],
        # the champion's own threshold for the same alert budget on the same validation days
        "champion": evaluate.threshold_for_alert_budget(
            valid, old_model.predict_proba(valid)[:, 1], budget
        ),
    }
    scores = {
        "challenger": new_model.predict_proba(test)[:, 1],
        "champion": old_model.predict_proba(test)[:, 1],
    }
    again = training.compare_with_champion(test, scores, thresholds, budget)
    display(pd.DataFrame({"recorded by the retrain": g, "recomputed here": again}).round(4))

## 5. The audit trail

`ml.ops_audit` has one row per event: what was asked (`event`), what happened (`outcome`), which versions, who decided and their comment, the drift check that led to it, and the GitHub run.

| event | outcome | written by |
|---|---|---|
| `retrain` | `challenger` or `failed_gates` | the training job, after an approved retrain request |
| `promote` | `promoted` | job `model_ops`, after an approved promotion request |
| `rollback` | `rolled_back` | job `model_ops`, after an approved rollback request |
| any of the three | `rejected` | copied from GitHub by the next prod-run |

A rejected request never gets credentials for prod, so it can't write its own row; that is by design. GitHub keeps the review, and the next approved prod run copies it here.

In [ ]:
if spark.catalog.tableExists(f"{CATALOG}.ml.ops_audit"):
    display(
        spark.sql(f"""
        SELECT a.event_ts, a.event, a.outcome, a.version, a.from_version, a.actor, a.reason,
               a.episode, c.status AS check_status, c.breaches AS check_breaches, a.run_url
        FROM {CATALOG}.ml.ops_audit a
        LEFT JOIN {CATALOG}.monitoring.drift_checks c ON c.check_id = a.check_id
        ORDER BY a.event_ts
    """)
    )
else:
    print(f"No decision has been recorded in {CATALOG} yet.")

## 6. What a promotion changes, and what it never does

A promotion moves `@champion` to the new version and `@previous_champion` to the old one. A rollback moves them back. Nothing is copied or rebuilt.

- **From the next scoring run**, new patient-hours are scored by the new champion, at its own alert threshold; the drift check judges it against its own data window.
- **Hours already scored keep the version that scored them.** `gold.risk_scores` is a log of what the ward saw; rescoring the past with a model that wasn't live would rewrite history and make monitoring compare the wrong predictions with outcomes.

After a promotion the log therefore holds more than one version:

In [ ]:
display(
    spark.sql(f"""
    SELECT model_version, count(*) AS hours, min(prediction_ts) AS first_hour,
           max(prediction_ts) AS last_hour,
           round(avg(CASE WHEN risk_band = 'High' THEN 1 ELSE 0 END) * 100, 1) AS pct_high_hours
    FROM {CATALOG}.gold.risk_scores GROUP BY model_version ORDER BY first_hour
""")
)

In [ ]:
# Try it yourself: 1. In section 4, give the champion its deployed threshold, not a fresh one.
# Which gate moves?

In [ ]:
# Try it yourself: 2. Run the comparison with n_boot=50, then 1000. How stable are the shares?

In [ ]:
# Try it yourself: 3. From ml.ops_audit and monitoring.drift_checks, list each act check
# and what was decided.

**Pitfall: retraining as a reflex.** Drift says something changed, not that a new model will be better. A sensor fault needs the sensors fixed; a week after a protocol change a retrain may still fail its gates. The gates exist so that a model which isn't better is not put in use.

**Pitfall: judging on a few events.** Twenty escalations can't carry a point comparison. Resample, compare both models on the same resamples, and look at the share of them each model wins.

**Pitfall: a gate tuned on the data it will judge.** These limits were designed on an offline rehearsal, so its pass rates describe the design. Prod's episodes use seeds no rehearsal used, and the limits were fixed in code before they ran.

**Next:** the Model & Drift dashboard, which puts the checks, the decisions and both models' numbers on one page.